# Instationary incompressible control problems

In this notebook, we describe how to solve instationary incompressible problems with the module control.

## An instationary incompressible Navier–Stokes control problem

Given $\beta>0$ and $t_f>0$, we consider the following instationary incompressible Navier–Stokes control problem:

$$
\min_{\vec{v},\vec{u}} \frac{1}{2} \int_0^{t_f} \| \vec{v} - \vec{v}_d \| ^2 + \frac{1}{\beta} \int_0^{t_f} \| \vec{u} \| ^2
$$

subject to:

$$
    \frac{\partial \vec{v}}{\partial t} - \nu \nabla^2 \; \vec{v} + \vec{v}\cdot \nabla \vec{v} + \nabla p = \vec{f} + \vec{u} \qquad \mathrm{in} \; \Omega \times (0, t_f), 
$$
$$
    - \nabla \cdot \vec{v} = 0 \qquad \mathrm{in} \; \Omega,
$$
$$
    \vec{v}(\mathbf{x},t) = \vec{g} \qquad \mathrm{on} \; \partial \Omega \times (0, t_f).
$$
$$
    \vec{v}(\mathbf{x},0) = \vec{v}_0 \qquad \mathrm{in} \; \Omega,
$$

For this example, we consider a time-dependent version of the lid-driven cavity in $\Omega = (-1, 1)$, with $\vec{v}_0 = [0, 0]^\top$, $\vec{f}=[0, 0]^\top$, and

$$
    \vec{g} = [t, 0]^\top \quad \mathrm{on} \; \partial \Omega_1 \times (0, 1),
$$
$$
    \vec{g} = [1, 0]^\top \quad \mathrm{on} \; \partial \Omega_1 \times (1, t_f),
$$
$$
    \vec{g} = [0, 0]^\top \quad \mathrm{on} \; \partial \Omega_2 \times (0, t_f),
$$

where we set $\partial \Omega_1:=(-1,1) \times \{ 1\}$ and $\partial \Omega_2 := \partial \Omega \setminus \partial \Omega_1$. We seek the desired state $\vec{v}_d = [0, 0]^\top$, and integrate the problem up to $t_f=2$ with trapezi in time. For the problem here, we set $\nu=\frac{1}{250}$ and $\beta=10^{-4}$.

The user has to construct the problem with the module Instationary, the call the module incompressible_non_linear_solve(). The kwargs and the default options are the same as for the stationary case. 

In [ ]:
from firedrake import *
from control.preconditioner import ConstantNullspace
from control.control import Instationary

beta = 1.0e-4

mesh = RectangleMesh(10, 10, 1.0, 1.0, originX=-1.0, originY=-1.0)

space_v = VectorFunctionSpace(mesh, "Lagrange", 2)
space_p = FunctionSpace(mesh, "Lagrange", 1)

n_t = 5
time_interval = (0.0, 2.0)


def my_DirichletBC_t_v(space_v, t):
    if float(t) < 1.0:
        my_bcs = [DirichletBC(space_v, Constant((t, 0.0)), (4,)),
                  DirichletBC(space_v, 0.0, (1, 2, 3))]
    else:
        my_bcs = [DirichletBC(space_v, Constant((1.0, 0.0)), (4,)),
                  DirichletBC(space_v, 0.0, (1, 2, 3))]

    return my_bcs


def forw_diff_operator_v(trial, test, u, t):
    # spatial differential for the forward problem
    nu = 1.0 / 250.0
    return (
        nu * inner(grad(trial), grad(test)) * dx
        + inner(dot(grad(trial), u), test) * dx)


def desired_state_v(test, t):
    space = test.function_space()

    v_d = Function(space, name="v_d")
    v_d.zero()

    return inner(v_d, test) * dx, v_d


def initial_condition_v(test):
    space = test.function_space()

    v_0 = Function(space)
    v_0.interpolate(as_vector([0.0, 0.0]))

    return v_0


def force_f_v(test, t):
    space = test.function_space()

    # force function
    f = Function(space)
    f.interpolate(as_vector([0.0, 0.0]))

    return inner(f, test) * dx


instationary_Navier_Stokes = Instationary(
    space_v, forw_diff_operator_v, desired_state=desired_state_v,
    force_function=force_f_v, beta=beta, initial_condition=initial_condition_v,
    time_interval=time_interval, n_t=n_t,
    bcs_v=my_DirichletBC_t_v)

# employing Chebyshev for the (1,1)-block
e_min_v = 0.3924
e_max_v = 2.0598

sp_11block = {
    "ksp_type": "chebyshev",
    "pc_type": "jacobi",
    "ksp_chebyshev_eigenvalues": f"{e_min_v:.16e}, {e_max_v:.16e}",
    "ksp_chebyshev_esteig": "0.0,0.0,0.0,0.0",
    "ksp_chebyshev_esteig_steps": 0,
    "ksp_chebyshev_esteig_noisy": False,
    "ksp_max_it": 20,
    "ksp_atol": 0.0,
    "ksp_rtol": 0.0}

# employing Chebyshev for the pressure-mass matrix
e_min_p = 0.5
e_max_p = 2.0
sp_M_p = {
    "ksp_type": "chebyshev",
    "pc_type": "jacobi",
    "ksp_chebyshev_eigenvalues": f"{e_min_p:.16e}, {e_max_p:.16e}",
    "ksp_chebyshev_esteig": "0.0,0.0,0.0,0.0",
    "ksp_chebyshev_esteig_steps": 0,
    "ksp_chebyshev_esteig_noisy": False,
    "ksp_max_it": 20,
    "ksp_atol": 0.0,
    "ksp_rtol": 0.0}

auxiliary_sp = {"sp_11block": sp_11block,
                "sp_M_p": sp_M_p}

solver_parameters = {"linear_solver": "fgmres",
                     "fgmres_restart": 10,
                     "maximum_iterations": 100,
                     "relative_tolerance": 1.0e-6,
                     "absolute_tolerance": 1.0e-6,
                     "monitor_convergence": True}

instationary_Navier_Stokes.incompressible_non_linear_solve(
    ConstantNullspace(), space_p=space_p,
    solver_parameters=solver_parameters,
    auxiliary_sp=auxiliary_sp, absolute_non_linear_tol=1.0e-5,
    max_non_linear_iter=10, create_output=False)